# CUDA dev notebook — Sensor Anomaly Detector

Runtime: Runtime > Change runtime type > GPU (T4 is fine for now).

Workflow: write/compile/run kernels here, then copy finished `.cu`/`.py` files back into the GitHub repo locally and commit from your machine.

In [ ]:
!nvcc --version
!nvidia-smi

## Clone the repo (so kernels stay versioned)

In [ ]:
!git clone https://github.com/nrwre/cudaproject.git
%cd cudaproject

## Compile + run: vector_add sanity check

In [ ]:
!nvcc -O2 cuda/vector_add.cu -o vector_add
!./vector_add

## Week 2: numeric validation

Generate one synthetic dataset, feed the *same* file to the CUDA kernel and the NumPy reference, then diff their anomaly flags exactly. This is the only way to actually prove correctness — comparing separately-generated random data proves nothing.

In [ ]:
!python python/generate_data.py data.bin

In [ ]:
!nvcc -O2 cuda/rolling_zscore.cu cuda/kernels.cu -o rolling_zscore
!./rolling_zscore data.bin gpu_anomalies.bin

In [ ]:
!python python/cpu_baseline.py data.bin cpu_anomalies.bin

In [ ]:
!python python/validate.py data.bin cpu_anomalies.bin gpu_anomalies.bin

Expect 0 mismatches (or a tiny number from float32 summation-order differences right at the threshold boundary — if that happens, it's worth understanding why, not just accepting it).

## Week 2: memory access pattern (sensor-major vs time-major)

Same algorithm, same thread-per-sensor mapping. v1 reads are strided across a warp (uncoalesced); v2 transposes to time-major so a warp's simultaneous reads are contiguous (coalesced). Compare the kernel-time lines in stderr output above/below.

In [ ]:
!nvcc -O2 cuda/rolling_zscore_coalesced.cu cuda/kernels.cu -o rolling_zscore_coalesced
!./rolling_zscore_coalesced data.bin gpu_anomalies_coalesced.bin

In [ ]:
# Validate the coalesced kernel against the same NumPy reference, then compare timings.
!python python/validate.py data.bin cpu_anomalies.bin gpu_anomalies_coalesced.bin

## Bigger dataset (the gap should widen with more sensors)

Memory-bound kernels show the coalescing gap more clearly at scale. Try generating a larger dataset and re-running both kernels to see how the timing gap changes.

In [ ]:
import sys
sys.path.insert(0, 'python')
from generate_data import generate, write_input_file

data, window, threshold = generate(num_sensors=65536, num_timesteps=2048)
write_input_file('data_large.bin', data, window, threshold)
print('wrote data_large.bin')

In [ ]:
!./rolling_zscore data_large.bin gpu_anomalies_large.bin
!./rolling_zscore_coalesced data_large.bin gpu_anomalies_large_coalesced.bin

## Week 3: pybind11 extension + CPU vs GPU benchmark

Build the kernels as a Python extension module so `backend/main.py` (FastAPI) can call them directly, and run the full CPU-vs-GPU benchmark sweep that produces the headline numbers for the README.

In [ ]:
!pip install pybind11 -q

In [ ]:
import pybind11, sysconfig
pybind11_includes = f"-I{pybind11.get_include()}"
python_includes = f"-I{sysconfig.get_path('include')}"
ext_suffix = sysconfig.get_config_var('EXT_SUFFIX')
print(pybind11_includes, python_includes, ext_suffix)

!nvcc -O2 -shared -Xcompiler -fPIC {pybind11_includes} {python_includes} \
    cuda/kernels.cu cuda/bindings.cpp -o anomaly_gpu{ext_suffix}

In [ ]:
import importlib, sys
sys.path.insert(0, '.')
import anomaly_gpu
importlib.reload(anomaly_gpu)
print(anomaly_gpu.__doc__)
print(dir(anomaly_gpu))

In [ ]:
# Sanity check the extension against the CUDA executable's output on the same file,
# before trusting it for the benchmark.
import sys
sys.path.insert(0, 'python')
from cpu_baseline import read_input_file
data, window, threshold = read_input_file('data.bin')
anomalies, kernel_ms = anomaly_gpu.rolling_zscore_coalesced(data, window, threshold)
print('pybind11 extension kernel_ms:', kernel_ms, 'flagged:', int(anomalies.sum()))
# Compare against gpu_anomalies_coalesced.bin produced by the standalone executable earlier
from cpu_baseline import write_output_file
write_output_file('gpu_anomalies_pybind.bin', anomalies)
!python python/validate.py data.bin cpu_anomalies.bin gpu_anomalies_pybind.bin

### Run the full CPU vs GPU benchmark sweep

This produces the real numbers for the README's Results table.

In [ ]:
!python python/benchmark.py benchmark_results.json
!cat benchmark_results.json

## Week 3: FastAPI backend smoke test

Confirms the backend picks up the GPU extension correctly in this environment, end to end, without needing a separate server process.

In [ ]:
!pip install fastapi uvicorn httpx -q
import sys
sys.path.insert(0, 'backend')
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)
print(client.get('/health').json())

resp = client.post('/run', json={'num_sensors': 4096, 'num_timesteps': 2048, 'window': 32, 'threshold': 3.0})
body = resp.json()
print('status:', resp.status_code)
print('gpu_available:', body['gpu_available'])
print('cpu_ms:', body['cpu_ms'], 'gpu_ms:', body['gpu_ms'], 'speedup_x:', body['speedup_x'])
print('flagged sensors:', len(body['flagged_sensors']))

## Pull changes back

Download any `.cu`/`.py` files you edited here (left sidebar > file browser > right-click > Download), overwrite them locally, then commit + push from your machine as usual.